In [ ]:
import pandas as pd
from prophet import Prophet
import numpy as np
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
data_path = Path('data/data_collection.csv')
if not data_path.exists():
    data_path = Path('../data/data_collection.csv')
if not data_path.is_file():
    raise FileNotFoundError('Cannot find data/data_collection.csv. Run from the repository root or notebooks/ directory.')
data = pd.read_csv(data_path)

In [ ]:
data_cleaned = data.apply(pd.to_numeric, errors='coerce').drop(columns=['column_7', 'column_14', 'column_31'])
data_cleaned.fillna(data_cleaned.mean(), inplace=True)

In [ ]:
df_prophet = pd.DataFrame({
    'ds': pd.date_range(start='2020-01-01', periods=len(data_cleaned), freq='D'),
    'y': data_cleaned['sugar_price']
})

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
mae_scores = []
rmse_scores = []

for train_index, test_index in tscv.split(df_prophet):
    train = df_prophet.iloc[train_index]
    test = df_prophet.iloc[test_index]
    
    model = Prophet(changepoint_prior_scale=0.05, seasonality_prior_scale=10)
    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test))
    forecast = model.predict(future)
    
    y_true = test['y'].values
    y_pred = forecast.tail(len(test))['yhat'].values
    
    mae_scores.append(mean_absolute_error(y_true, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_true, y_pred)))

print(f"Average MAE: {np.mean(mae_scores)}")
print(f"Average RMSE: {np.mean(rmse_scores)}")

In [ ]:
final_model = Prophet(changepoint_prior_scale=0.05, seasonality_prior_scale=10)
final_model.fit(df_prophet)

In [ ]:
future = final_model.make_future_dataframe(periods=365)
forecast = final_model.predict(future)

In [ ]:
forecast_values = forecast['yhat'][-365:]

In [ ]:
submission = pd.DataFrame({'ID': np.arange(1, 366), 'Prediction': forecast_values})
submission.to_csv('submission.csv', index=False)

In [ ]:
#print(submission.head(365))